# Stochastic Processes with RustQuant

RustQuant provides a rich library of stochastic processes commonly used in
quantitative finance, including:

- **Brownian motions**: Standard, Arithmetic, Geometric, Fractional
- **Short-rate models**: CIR, Vasicek, Hull-White, Ho-Lee, Black-Derman-Toy
- **Stochastic volatility**: Heston, SABR, CEV
- **Jump-diffusion**: Merton

All processes are simulated using the Euler-Maruyama scheme.

## Setup

In [ ]:
:dep RustQuant = { path = "../crates/RustQuant" }

In [ ]:
use RustQuant::stochastics::*;

## 1. Geometric Brownian Motion (GBM)

The standard model for stock prices:

$$dS_t = \mu S_t \, dt + \sigma S_t \, dW_t$$

where $\mu$ is the drift and $\sigma$ is the volatility.

In [ ]:
let gbm = GeometricBrownianMotion::new(0.05, 0.20);

let config = StochasticProcessConfig::new(
    100.0,  // initial value
    0.0,    // start time
    1.0,    // end time (1 year)
    252,    // daily steps
    StochasticScheme::EulerMaruyama,
    5,      // 5 simulations
    false,  // sequential
    None,
);

let output = gbm.generate(&config);

println!("GBM: {} paths, {} time steps each", output.paths.len(), output.paths[0].len());
for (i, path) in output.paths.iter().enumerate() {
    println!("  Path {}: start = {:.2}, end = {:.2}", i + 1, path[0], path[path.len() - 1]);
}

## 2. Arithmetic Brownian Motion (ABM)

$$dX_t = \mu \, dt + \sigma \, dW_t$$

Useful for modeling spreads or log-returns.

In [ ]:
let abm = ArithmeticBrownianMotion::new(0.05, 0.20);
let output = abm.generate(&config);

println!("ABM paths:");
for (i, path) in output.paths.iter().enumerate() {
    println!("  Path {}: start = {:.2}, end = {:.2}", i + 1, path[0], path[path.len() - 1]);
}

## 3. Ornstein-Uhlenbeck Process

$$dX_t = \theta(\mu - X_t) \, dt + \sigma \, dW_t$$

A mean-reverting process, widely used for interest rates and pairs trading.

In [ ]:
let ou = OrnsteinUhlenbeck::new(
    0.5,   // mu (long-term mean)
    2.0,   // theta (mean-reversion speed)
    0.1,   // sigma (volatility)
);

let ou_config = StochasticProcessConfig::new(
    0.5, 0.0, 5.0, 1000, StochasticScheme::EulerMaruyama, 3, false, None,
);

let output = ou.generate(&ou_config);

println!("Ornstein-Uhlenbeck (mean-reverting to 0.5):");
for (i, path) in output.paths.iter().enumerate() {
    let mean: f64 = path.iter().sum::<f64>() / path.len() as f64;
    println!("  Path {}: mean = {:.4}, final = {:.4}", i + 1, mean, path[path.len() - 1]);
}

## 4. Cox-Ingersoll-Ross (CIR)

$$dr_t = \kappa(\theta - r_t) \, dt + \sigma \sqrt{r_t} \, dW_t$$

A mean-reverting process that stays non-negative (when $2\kappa\theta > \sigma^2$).
Commonly used for short-rate and stochastic volatility models.

In [ ]:
let cir = CoxIngersollRoss::new(
    0.05,  // theta (long-term mean rate)
    0.9,   // kappa (mean-reversion speed)
    0.1,   // sigma (vol of vol)
);

let rate_config = StochasticProcessConfig::new(
    0.03, 0.0, 10.0, 2520, StochasticScheme::EulerMaruyama, 3, false, None,
);

let output = cir.generate(&rate_config);

println!("CIR Short-Rate Model (mean-reverting to 0.05):");
for (i, path) in output.paths.iter().enumerate() {
    let min = path.iter().cloned().fold(f64::INFINITY, f64::min);
    let max = path.iter().cloned().fold(f64::NEG_INFINITY, f64::max);
    println!("  Path {}: final = {:.4}, min = {:.4}, max = {:.4}", i + 1, path[path.len() - 1], min, max);
}

## 5. Hull-White Model

$$dr_t = (\theta(t) - \alpha \, r_t) \, dt + \sigma \, dW_t$$

An extension of the Vasicek model with time-dependent parameters.

In [ ]:
let hw = HullWhite::new(0.1, 0.2, 0.01);
let output = hw.generate(&rate_config);

println!("Hull-White Short-Rate paths:");
for (i, path) in output.paths.iter().enumerate() {
    println!("  Path {}: start = {:.4}, end = {:.4}", i + 1, path[0], path[path.len() - 1]);
}

## 6. Merton Jump Diffusion

$$dS_t = (\mu - \lambda k) S_t \, dt + \sigma S_t \, dW_t + J_t S_t \, dN_t$$

GBM with added Poisson jumps for modeling sudden market moves.

In [ ]:
let mjd = MertonJumpDiffusion::new(
    0.05,   // drift
    0.20,   // diffusion volatility
    5.0,    // jump intensity (lambda)
    -0.02,  // mean jump size
    0.10,   // jump size volatility
);

let mjd_config = StochasticProcessConfig::new(
    100.0, 0.0, 1.0, 252, StochasticScheme::EulerMaruyama, 3, false, None,
);

let output = mjd.generate(&mjd_config);

println!("Merton Jump Diffusion:");
for (i, path) in output.paths.iter().enumerate() {
    println!("  Path {}: start = {:.2}, end = {:.2}", i + 1, path[0], path[path.len() - 1]);
}

## 7. Fractional Brownian Motion

$$B^H_t$$

A generalization of Brownian motion with Hurst parameter $H \in (0,1)$:
- $H = 0.5$: Standard Brownian motion
- $H > 0.5$: Long-range dependence (trending)
- $H < 0.5$: Anti-persistent (mean-reverting)

In [ ]:
let fbm_trending = FractionalBrownianMotion::new(0.8, FractionalProcessGeneratorMethod::FFT);
let fbm_reverting = FractionalBrownianMotion::new(0.2, FractionalProcessGeneratorMethod::FFT);

let fbm_config = StochasticProcessConfig::new(
    0.0, 0.0, 1.0, 252, StochasticScheme::EulerMaruyama, 1, false, None,
);

let trending = fbm_trending.generate(&fbm_config);
let reverting = fbm_reverting.generate(&fbm_config);

println!("Fractional BM (H=0.8, trending):     final = {:.4}", trending.paths[0].last().unwrap());
println!("Fractional BM (H=0.2, mean-reverting): final = {:.4}", reverting.paths[0].last().unwrap());

## 8. Multiple Simulations

Generate many paths for Monte Carlo analysis, with optional parallel execution.

In [ ]:
let gbm = GeometricBrownianMotion::new(0.05, 0.20);

let mc_config = StochasticProcessConfig::new(
    100.0, 0.0, 1.0, 252, StochasticScheme::EulerMaruyama,
    10_000,  // 10,000 paths
    true,    // parallel execution
    None,
);

let output = gbm.generate(&mc_config);

// Compute terminal distribution statistics
let terminals: Vec<f64> = output.paths.iter().map(|p| *p.last().unwrap()).collect();
let mean = terminals.iter().sum::<f64>() / terminals.len() as f64;
let variance = terminals.iter().map(|x| (x - mean).powi(2)).sum::<f64>() / terminals.len() as f64;

println!("GBM Monte Carlo ({} paths):", terminals.len());
println!("  Mean terminal value:  {:.2}", mean);
println!("  Std dev:              {:.2}", variance.sqrt());
println!("  Expected (analytic):  {:.2}", 100.0 * (0.05_f64 * 1.0).exp());

## Summary of Available Processes

| Process | Type | Key Parameters |
|---------|------|----------------|
| `BrownianMotion` | Diffusion | - |
| `ArithmeticBrownianMotion` | Diffusion | mu, sigma |
| `GeometricBrownianMotion` | Diffusion | mu, sigma |
| `OrnsteinUhlenbeck` | Mean-reverting | mu, theta, sigma |
| `CoxIngersollRoss` | Mean-reverting (non-negative) | theta, kappa, sigma |
| `HullWhite` | Short-rate | alpha, sigma, theta |
| `ExtendedVasicek` | Short-rate | alpha, sigma, theta |
| `HoLee` | Short-rate | sigma, theta |
| `BlackDermanToy` | Short-rate | sigma, theta |
| `MertonJumpDiffusion` | Jump-diffusion | mu, sigma, lambda, mu_j, sigma_j |
| `FractionalBrownianMotion` | Long memory | Hurst parameter |
| `GeometricBrownianBridge` | Bridge | mu, sigma, target, T |
| `ConstantElasticityOfVariance` | Local vol | mu, sigma, elasticity |